### Importing functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *
from time import sleep

### DATA READING

In [0]:
df = spark.read.format('parquet').load('abfss://bronze@databricksretailproject.dfs.core.windows.net/customers')



In [0]:
df = df.drop('rescued_data')

In [0]:
df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
df = df.withColumn('domain', split(col('email'),'@')[1])


In [0]:
df.groupBy('domain').agg(count('customer_id').alias('count')).orderBy(col('count').desc()).display()

domain,count
gmail.com,374
hotmail.com,360
yahoo.com,331
brown.com,8
davis.com,8
smith.com,7
hernandez.com,5
johnson.com,5
kennedy.com,4
miller.com,4


In [0]:
df_yahoo = df.filter(col('domain') == 'yahoo.com')

df_gmail = df.filter(col('domain') == 'gmail.com')

df_hotmail = df.filter(col('domain') == 'hotmail.com')


print('gmail.com')
#df_gmail.display()
sleep(5)
#print('yahoo.com')
df_yahoo.display()
sleep(5)
#print('hotmail.com')
df_hotmail.display()
sleep(5)


gmail.com


customer_id,first_name,last_name,email,city,state,_rescued_data,domain
C00003,Craig,Hayes,rebeccamiller@yahoo.com,South Stephenshire,LA,null,yahoo.com
C00005,Sean,Vasquez,carrie45@yahoo.com,East Dennistown,RI,null,yahoo.com
C00009,Mary,Green,dennis03@yahoo.com,Kimberlyview,MD,null,yahoo.com
C00014,Thomas,Hartman,lmoore@yahoo.com,East Tiffanybury,MS,null,yahoo.com
C00024,Heather,Owens,andreawilliams@yahoo.com,Stewartport,NM,null,yahoo.com
C00025,Christina,Bennett,dawn65@yahoo.com,Port Juliebury,IN,null,yahoo.com
C00026,Mark,Harris,rebecca20@yahoo.com,Ortizshire,WY,null,yahoo.com
C00042,Jenna,Lewis,hannasteven@yahoo.com,South Garrettton,IA,null,yahoo.com
C00048,Natalie,Gentry,floresvanessa@yahoo.com,Lake Andrea,MO,null,yahoo.com
C00051,Frank,Phillips,millerjennifer@yahoo.com,Anthonymouth,CO,null,yahoo.com


customer_id,first_name,last_name,email,city,state,_rescued_data,domain
C00015,Diane,Harris,daniellowe@hotmail.com,South Courtney,SC,null,hotmail.com
C00020,Juan,Collins,rojassandra@hotmail.com,South Oscar,NV,null,hotmail.com
C00031,Jeanette,Smith,kramerkaylee@hotmail.com,Travisview,CT,null,hotmail.com
C00033,Carrie,Wheeler,lindasummers@hotmail.com,South Andrewchester,CO,null,hotmail.com
C00036,Jacob,Mccoy,jesustaylor@hotmail.com,Port Nicoleborough,AR,null,hotmail.com
C00037,Holly,Arnold,eric78@hotmail.com,Millerfurt,VT,null,hotmail.com
C00038,Robert,Haas,ythomas@hotmail.com,Port Michaelstad,MS,null,hotmail.com
C00039,Teresa,Ward,angela55@hotmail.com,Lake Michaelhaven,AL,null,hotmail.com
C00041,James,Brooks,pbarnes@hotmail.com,Allisonhaven,AK,null,hotmail.com
C00043,Margaret,Sullivan,ronnie19@hotmail.com,South Johnville,OK,null,hotmail.com


In [0]:
df = df.withColumn('fullname', concat(col('first_name'), lit(' '), col('last_name')))
df = df.drop('first_name', 'last_name')



In [0]:
df.write.format('delta').mode('append').save('abfss://silver@databricksretailproject.dfs.core.windows.net/customers')



In [0]:
%sql
create schema if not exists retail_databricks_catalog.silver

In [0]:
%sql
create table if not exists retail_databricks_catalog.silver.customer_silver 

using delta

location 'abfss://silver@databricksretailproject.dfs.core.windows.net/customers'

In [0]:
%sql
create table if not exists retail_databricks_catalog.silver.products_silver 

using delta

location 'abfss://silver@databricksretailproject.dfs.core.windows.net/products'

In [0]:
%sql
create table if not exists retail_databricks_catalog.silver.orders_silver 

using delta

location 'abfss://silver@databricksretailproject.dfs.core.windows.net/orders'

In [0]:
%sql
create table if not exists retail_databricks_catalog.silver.regions_silver 

using delta

location 'abfss://silver@databricksretailproject.dfs.core.windows.net/regions'